# HealthData.gov - US Health System Data

This notebook demonstrates how to access US health system data from [HealthData.gov](https://healthdata.gov/), including hospital capacity, COVID-19 metrics, and vaccination data.

## Data Source
- **Provider**: US Department of Health and Human Services (HHS)
- **Coverage**: United States (all states and territories)
- **License**: Public Domain (US Government)
- **API**: Socrata Open Data API (SODA)
- **Update Frequency**: Daily to Weekly

## Available Datasets
- Hospital capacity by state
- COVID-19 patient impact
- Vaccination data
- Testing data
- Nursing home data

In [ ]:
import sys
sys.path.insert(0, '../../')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from scripts.accessors.healthdata_gov import HealthDataGovAccessor

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Initialize accessor
hdg = HealthDataGovAccessor()

## 1. List Available Datasets

In [ ]:
# Display available datasets
datasets = hdg.list_datasets()
datasets

## 2. Hospital Capacity Data

In [ ]:
# Get hospital capacity for California (last 30 days)
end_date = datetime.now().strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d")

print(f"Fetching CA hospital data from {start_date} to {end_date}...")

ca_hospitals = hdg.get_hospital_capacity(
    state="CA",
    date_range=(start_date, end_date)
)

print(f"Retrieved {len(ca_hospitals)} records")
ca_hospitals.head()

In [ ]:
# Plot hospital capacity over time
if not ca_hospitals.empty and "date" in ca_hospitals.columns:
    fig, axes = plt.subplots(2, 1, figsize=(12, 10))
    
    # Total beds used
    if "inpatient_beds_used" in ca_hospitals.columns:
        axes[0].plot(ca_hospitals["date"], ca_hospitals["inpatient_beds_used"], 
                     marker="o", linewidth=2, label="Inpatient Beds Used")
        axes[0].set_title("California Hospital Bed Utilization")
        axes[0].set_ylabel("Beds Used")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
    
    # COVID beds
    if "inpatient_beds_used_covid" in ca_hospitals.columns:
        axes[1].plot(ca_hospitals["date"], ca_hospitals["inpatient_beds_used_covid"],
                     marker="o", color="red", linewidth=2, label="COVID Beds")
        axes[1].set_title("California COVID-19 Hospitalizations")
        axes[1].set_ylabel("COVID Beds Used")
        axes[1].set_xlabel("Date")
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 3. COVID-19 Metrics

In [ ]:
# Get COVID-19 metrics for New York
ny_covid = hdg.get_covid_metrics(
    state="NY",
    date_range=(start_date, end_date)
)

print(f"Retrieved {len(ny_covid)} records")
ny_covid.head()

In [ ]:
# Plot COVID metrics
if not ny_covid.empty:
    plt.figure(figsize=(12, 6))
    
    covid_cols = [col for col in ny_covid.columns if "covid" in col.lower()]
    for col in covid_cols[:3]:  # Plot first 3 COVID metrics
        if ny_covid[col].notna().any():
            plt.plot(ny_covid["date"], ny_covid[col], marker="o", label=col)
    
    plt.title("New York COVID-19 Hospital Metrics")
    plt.xlabel("Date")
    plt.ylabel("Count")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. State Comparison

In [ ]:
# Compare hospital capacity across multiple states
states = ["CA", "TX", "NY", "FL"]

comparison = hdg.compare_states(
    states=states,
    metric="inpatient_beds_used_covid",
    date=start_date
)

comparison

In [ ]:
# Visualize state comparison
if not comparison.empty and "inpatient_beds_used_covid" in comparison.columns:
    plt.figure(figsize=(10, 6))
    
    states_clean = comparison[comparison["inpatient_beds_used_covid"].notna()]
    plt.bar(states_clean["state"], states_clean["inpatient_beds_used_covid"])
    
    plt.title(f"COVID-19 Hospital Beds by State ({start_date})")
    plt.xlabel("State")
    plt.ylabel("COVID Beds Used")
    plt.tight_layout()
    plt.show()

## 5. Current Hospital Statistics

In [ ]:
# Get current stats for selected states
states = ["CA", "NY", "TX", "FL", "IL"]

current_stats = []
for state in states:
    try:
        stats = hdg.get_current_hospital_stats(state=state)
        current_stats.append(stats)
    except Exception as e:
        print(f"Error getting {state}: {e}")

stats_df = pd.DataFrame(current_stats)
stats_df

## 6. Vaccination Data

In [ ]:
# Get vaccination data
vax_data = hdg.get_vaccination_data(
    state="CA",
    date_range=(start_date, end_date)
)

print(f"Retrieved {len(vax_data)} records")
vax_data.head()

In [ ]:
# Plot vaccination progress
if not vax_data.empty:
    # Try to find relevant columns
    vax_cols = [col for col in vax_data.columns if any(x in col.lower() 
                 for x in ["dose", "series", "complete", "admin"])]
    
    if vax_cols:
        plt.figure(figsize=(12, 6))
        for col in vax_cols[:3]:
            if vax_data[col].dtype in ['int64', 'float64']:
                plt.plot(vax_data["date"], vax_data[col], marker="o", label=col[:30])
        
        plt.title("California Vaccination Progress")
        plt.xlabel("Date")
        plt.ylabel("Count")
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

## 7. Testing Data

In [ ]:
# Get COVID-19 testing data
testing_data = hdg.get_testing_data(
    state="NY",
    date_range=(start_date, end_date)
)

print(f"Retrieved {len(testing_data)} records")
testing_data.head()

In [ ]:
# Plot testing trends
if not testing_data.empty:
    test_cols = [col for col in testing_data.columns if any(x in col.lower() 
                  for x in ["test", "positive", "result"])]
    
    if test_cols:
        plt.figure(figsize=(12, 6))
        for col in test_cols[:2]:
            if testing_data[col].dtype in ['int64', 'float64']:
                plt.plot(testing_data["date"], testing_data[col], marker="o", label=col[:30])
        
        plt.title("New York COVID-19 Testing Data")
        plt.xlabel("Date")
        plt.ylabel("Count")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

## Summary

This notebook demonstrated how to:
1. List available HealthData.gov datasets
2. Retrieve hospital capacity data by state
3. Access COVID-19 patient impact metrics
4. Compare metrics across multiple states
5. Get current hospital statistics
6. Analyze vaccination progress
7. Work with testing data

For more information, visit: https://healthdata.gov/